# Battery Thermal Surrogate: Quick Start Guide

This notebook provides a quick introduction to the physics-informed thermal surrogate project.

## Workflow

1. Generate synthetic thermal simulation data
2. Explore the dataset
3. Train a simple model
4. Evaluate predictions

**Note**: This notebook uses small datasets for quick demonstration. For full-scale training, use the CLI scripts.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.physics.solver import HeatSolver2D
from src.physics.materials import create_material_mask, compute_signed_distance
from src.models import PCUNet
from src.utils.device import get_device

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

print(f"PyTorch version: {torch.__version__}")
print(f"Project root: {PROJECT_ROOT}")

## 1. Generate a Sample Trajectory

Let's create a simple material mask and run the physics solver.

In [ ]:
# Configuration
GRID_SIZE = 32
DX = DY = 1e-3  # 1 mm
T_AMB = 300.0   # Ambient temperature [K]
N_STEPS = 100

# Create material mask
mask = create_material_mask(grid_size=GRID_SIZE, layout="grid", n_cells=4)

# Visualize material layout
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(mask, cmap="tab10", origin="lower")
ax.set_title("Material Mask")
ax.set_xlabel("x (grid cells)")
ax.set_ylabel("y (grid cells)")
plt.colorbar(im, ax=ax, label="Material ID")
plt.tight_layout()
plt.show()

print(f"Battery cells: {(mask == 0).sum()} pixels")
print(f"Coolant: {(mask == 1).sum()} pixels")
print(f"Insulation: {(mask == 2).sum()} pixels")

In [ ]:
# Set up physics parameters
k_values = np.array([2.0, 0.6, 0.04])      # Conductivity
rho_values = np.array([2500.0, 998.0, 30.0])  # Density
cp_values = np.array([700.0, 4182.0, 1400.0]) # Specific heat

# Create solver
solver = HeatSolver2D.from_material_fields(
    mask=mask,
    k_values=k_values,
    rho_values=rho_values,
    cp_values=cp_values,
    dx=DX,
    dy=DY,
    dt=0.0,  # Will be set automatically
    T_amb=T_AMB,
    h_conv=50.0,
)

# Use stable time step
solver.dt = solver.max_stable_dt * 0.5
print(f"Time step: {solver.dt:.2e} s")
print(f"Stability check: {solver.check_stability()}")

In [ ]:
# Run simulation
T0 = np.full((GRID_SIZE, GRID_SIZE), T_AMB)
source_mask = (mask == 0).astype(np.float64)  # Heat only in battery cells

print("Running thermal simulation...")
trajectory = solver.solve(
    T0=T0,
    n_steps=N_STEPS,
    q0=5e5,  # Heat generation rate
    source_mask=source_mask,
    save_every=10,
)

print(f"Trajectory shape: {trajectory.shape}")
print(f"Temperature range: [{trajectory.min():.2f}, {trajectory.max():.2f}] K")

## 2. Visualize Temperature Evolution

In [ ]:
# Plot snapshots at different times
time_indices = [0, 3, 6, 9]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

vmin, vmax = trajectory.min(), trajectory.max()

for ax, t_idx in zip(axes, time_indices):
    im = ax.imshow(trajectory[t_idx], cmap="hot", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(f"t = {t_idx * 10} steps")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

fig.colorbar(im, ax=axes, label="Temperature [K]", fraction=0.046, pad=0.04)
fig.suptitle("Temperature Evolution", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Load and Explore Dataset

If you've generated a dataset, you can load and explore it here.

In [ ]:
# Check if dataset exists
data_path = PROJECT_ROOT / "data" / "raw" / "thermal_dataset.h5"

if data_path.exists():
    print(f"Dataset found: {data_path}")
    
    from src.data_utils.dataset import ThermalDataset
    
    dataset = ThermalDataset(data_path)
    print(f"Total samples: {len(dataset)}")
    
    # Get a sample
    sample = dataset[0]
    print(f"Input shape: {sample['input'].shape}")
    print(f"Target shape: {sample['target'].shape}")
    print(f"Physics params: {sample['physics']}")
    
    # Visualize input channels
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    channel_names = ["T", "Cell Mask", "Coolant Mask", "Insul. Mask", 
                    "k", "q", "h", "SDF Cell", "SDF Coolant"]
    
    for i, (ax, name) in enumerate(zip(axes.flat, channel_names)):
        ax.imshow(sample['input'][i].numpy(), cmap="viridis", origin="lower")
        ax.set_title(name)
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()
    
else:
    print(f"Dataset not found at {data_path}")
    print("Run: python scripts/generate_data.py --quick_test")

## 4. Test Neural Network Model

Create a small PC-U-Net and test it on random data.

In [ ]:
# Set up device
device = get_device(verbose=True)

# Create model
model = PCUNet(
    in_channels=9,
    out_channels=1,
    base_features=16,
    num_levels=3,
    dropout_rate=0.1,
).to(device)

print(f"Model has {model.count_parameters():,} parameters")

# Test forward pass
dummy_input = torch.randn(1, 9, GRID_SIZE, GRID_SIZE).to(device)
dummy_physics = torch.randn(1, 3).to(device)

with torch.no_grad():
    output = model(dummy_input, dummy_physics)

print(f"Output shape: {output.shape}")
print("Forward pass successful!")

## 5. Visualize Model Predictions (if trained model exists)

In [ ]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "best_model.pt"

if checkpoint_path.exists() and data_path.exists():
    print("Loading trained model...")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    
    # Get a sample from dataset
    sample = dataset[100]
    input_tensor = sample['input'].unsqueeze(0).to(device)
    physics_tensor = sample['physics'].unsqueeze(0).to(device)
    target = sample['target'].squeeze().numpy()
    
    # Predict
    with torch.no_grad():
        pred = model(input_tensor, physics_tensor)
    
    pred = pred.squeeze().cpu().numpy()
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    vmin = min(target.min(), pred.min())
    vmax = max(target.max(), pred.max())
    
    im0 = axes[0].imshow(target, cmap="hot", origin="lower", vmin=vmin, vmax=vmax)
    axes[0].set_title("Ground Truth")
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(pred, cmap="hot", origin="lower", vmin=vmin, vmax=vmax)
    axes[1].set_title("Prediction")
    plt.colorbar(im1, ax=axes[1])
    
    error = np.abs(pred - target)
    im2 = axes[2].imshow(error, cmap="Reds", origin="lower")
    axes[2].set_title(f"Error (MAE={error.mean():.3f} K)")
    plt.colorbar(im2, ax=axes[2])
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No trained model found.")
    print("Train a model with: python scripts/train.py --debug --epochs 5")

## Next Steps

1. **Generate full dataset**: `python scripts/generate_data.py`
2. **Train model**: `python scripts/train.py --config configs/train.yaml`
3. **Evaluate**: `python scripts/evaluate.py --model checkpoints/best_model.pt`
4. **Run tests**: `pytest tests/ -v`

See `README.md` for complete documentation!